In [1]:
%load_ext cudf.pandas
import sys, os

# point this at the directory *containing* tpch/
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))  
sys.path.insert(0, tpch_parent)

#from tpch import load_lineitem, q01
import pandas as pd
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}  # or load from JSON


In [2]:
# load just the datasets q01 needs:
def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [3]:
%%time
threshold = '1998-09-02'
cols = [
    'L_QUANTITY', 'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX',
    'L_RETURNFLAG', 'L_LINESTATUS', 'L_SHIPDATE', 'L_ORDERKEY'
]

# 1) Single .loc with bracket indexing for projection & filter
lf = lineitem.loc[lineitem['L_SHIPDATE'] <= threshold, cols]

# 2) Vectorized column computations in one assign call
lf = lf.assign(
    AVG_QTY=lf['L_QUANTITY'],
    AVG_PRICE=lf['L_EXTENDEDPRICE'],
    DISC_PRICE=lf['L_EXTENDEDPRICE'] * (1 - lf['L_DISCOUNT']),
    CHARGE=(
        lf['L_EXTENDEDPRICE'] * (1 - lf['L_DISCOUNT']) * (1 + lf['L_TAX'])
    )
)

# 3) GPU-friendly groupby & aggregation
total = lf.groupby(
    ['L_RETURNFLAG', 'L_LINESTATUS'], as_index=False
).agg({
    'L_QUANTITY': 'sum',
    'L_EXTENDEDPRICE': 'sum',
    'DISC_PRICE': 'sum',
    'CHARGE': 'sum',
    'AVG_QTY': 'mean',
    'AVG_PRICE': 'mean',
    'L_DISCOUNT': 'mean',
    'L_ORDERKEY': 'count'
})

In [4]:
%%time
#original

date = pd.Timestamp("1998-09-02")
lineitem_filtered = lineitem.loc[
    :,
    [
        "L_QUANTITY",
        "L_EXTENDEDPRICE",
        "L_DISCOUNT",
        "L_TAX",
        "L_RETURNFLAG",
        "L_LINESTATUS",
        "L_SHIPDATE",
        "L_ORDERKEY",
    ],
]
sel = lineitem_filtered.L_SHIPDATE <= date
lineitem_filtered = lineitem_filtered[sel]
lineitem_filtered["AVG_QTY"] = lineitem_filtered.L_QUANTITY
lineitem_filtered["AVG_PRICE"] = lineitem_filtered.L_EXTENDEDPRICE
lineitem_filtered["DISC_PRICE"] = lineitem_filtered.L_EXTENDEDPRICE * (
    1 - lineitem_filtered.L_DISCOUNT
)
lineitem_filtered["CHARGE"] = (
    lineitem_filtered.L_EXTENDEDPRICE
    * (1 - lineitem_filtered.L_DISCOUNT)
    * (1 + lineitem_filtered.L_TAX)
)
gb = lineitem_filtered.groupby(["L_RETURNFLAG", "L_LINESTATUS"], as_index=False)[
    [
        "L_QUANTITY",
        "L_EXTENDEDPRICE",
        "DISC_PRICE",
        "CHARGE",
        "AVG_QTY",
        "AVG_PRICE",
        "L_DISCOUNT",
        "L_ORDERKEY",
    ]
]
total = gb.agg(
    {
        "L_QUANTITY": "sum",
        "L_EXTENDEDPRICE": "sum",
        "DISC_PRICE": "sum",
        "CHARGE": "sum",
        "AVG_QTY": "mean",
        "AVG_PRICE": "mean",
        "L_DISCOUNT": "mean",
        "L_ORDERKEY": "count",
    }
)

CPU times: user 1.89 s, sys: 912 ms, total: 2.81 s
Wall time: 2.42 s


In [4]:
total.info( )

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   L_RETURNFLAG     4 non-null      object
 1   L_LINESTATUS     4 non-null      object
 2   L_QUANTITY       4 non-null      int64
 3   L_EXTENDEDPRICE  4 non-null      float64
 4   DISC_PRICE       4 non-null      float64
 5   CHARGE           4 non-null      float64
 6   AVG_QTY          4 non-null      float64
 7   AVG_PRICE        4 non-null      float64
 8   L_DISCOUNT       4 non-null      float64
 9   L_ORDERKEY       4 non-null      int64
dtypes: float64(6), int64(2), object(2)
memory usage: 304.0+ bytes


In [5]:
total

,L_RETURNFLAG,L_LINESTATUS,L_QUANTITY,L_EXTENDEDPRICE,DISC_PRICE,CHARGE,AVG_QTY,AVG_PRICE,L_DISCOUNT,L_ORDERKEY
0,A,F,37734107,5.658655e+10,5.375826e+10,5.590907e+10,25.522006,38273.129735,0.049985,1478493
1,N,F,991417,1.487505e+09,1.413082e+09,1.469649e+09,25.516472,38284.467761,0.050093,38854
2,N,O,74476040,1.117017e+11,1.061182e+11,1.103670e+11,25.502227,38249.117989,0.049997,2920374
3,R,F,37719753,5.656804e+10,5.374129e+10,5.588962e+10,25.505794,38250.854626,0.050009,1478870
